In [39]:
import pandas as pd
from pathlib import Path
DATA_DICTIONARY_PATH ="metadata/processed/data_dictionary.csv"
EXPORT_LONG_PATH = "data/processed/export_long_clean.csv"

OUTPUT_DIR = Path("results/dates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [40]:
dd=pd.read_csv(DATA_DICTIONARY_PATH, sep=";", dtype="object")
export_long=pd.read_csv(EXPORT_LONG_PATH, sep=";", dtype="object")

In [41]:
datetypes=dd[dd["data_type"]== "DATE"].copy()
datetypes

,Form,form_id,form_type,record_urn,record_designation,record_mandatory,record_repeatable,dataelement_urn,dataelement_designation,dataelement_definition,dataelement_mandatory,dataelement_repeatable,data_type,data_format,unit,source_id,slots
6,Screen data,222:8,Basic,NaN,NaN,NaN,NaN,urn:osse-11:dataelement:48:1,Date of informed consent signature:,Date of informed consent signature:,true,false,DATE,DD.MM.YYYY,NaN,X_222_0_48,NaN
48,Patient’s information and Medical history,225:2,Basic,NaN,NaN,NaN,NaN,urn:osse-11:dataelement:55:1,Date of birth,Date of birth,true,false,DATE,DD.MM.YYYY,NaN,X_225_0_55,NaN
54,Patient’s information and Medical history,225:2,Basic,urn:osse-11:record:22:1,Diagnosis of Cirrhosis,false,false,urn:osse-11:dataelement:97:1,Date of diagnosis of chronic liver disease,Date of diagnosis of chronic liver disease,NaN,NaN,DATE,DD.MM.YYYY,NaN,X_225_22_97,NaN
55,Patient’s information and Medical history,225:2,Basic,urn:osse-11:record:22:1,Diagnosis of Cirrhosis,false,false,urn:osse-11:dataelement:98:1,Date of diagnosis of cirrhosis,Date of diagnosis of cirrhosis,NaN,NaN,DATE,DD.MM.YYYY,NaN,X_225_22_98,NaN
57,Patient’s information and Medical history,225:2,Basic,urn:osse-11:record:22:1,Diagnosis of Cirrhosis,false,false,urn:osse-11:dataelement:100:1,If AD-Date of first acute decompensation,If AD-Date of first acute decompensation,NaN,NaN,DATE,MM.YYYY,NaN,X_225_22_100,NaN
60,Patient’s information and Medical history,225:2,Basic,urn:osse-11:record:22:1,Diagnosis of Cirrhosis,false,false,urn:osse-11:dataelement:103:1,If listed-listed liver transplantation date,If listed-listed liver transplantation date,NaN,NaN,DATE,DD.MM.YYYY,NaN,X_225_22_103,NaN
158,Data at visit,228:9,Longitudinal,urn:osse-11:record:47:1,Visit Date,false,false,urn:osse-11:dataelement:197:1,Visit Date,Visit Date,NaN,NaN,DATE,DD.MM.YYYY,NaN,X_228_47_197,NaN
182,Clinical features,229:3,Longitudinal,urn:osse-11:record:51:1,4. HVPG: hepatic-venous-pressure-gradient,false,false,urn:osse-11:dataelement:212:1,Date of HVPG,Date of HVPG,NaN,NaN,DATE,DD.MM.YYYY,NaN,X_229_51_212,NaN
205,Physical examination,230:9,Longitudinal,NaN,NaN,NaN,NaN,urn:osse-11:dataelement:232:1,Physical examination date,Physical examination date,true,false,DATE,DD.MM.YYYY,NaN,X_230_0_232,NaN
231,Laboratory data 1,231:12,Longitudinal,NaN,NaN,NaN,NaN,urn:osse-11:dataelement:242:1,Date of Collection,Date of Collection,false,false,DATE,DD.MM.YYYY,NaN,X_231_0_242,NaN


In [42]:
print(len(datetypes))
print(datetypes["data_format"].unique())
print(datetypes["data_format"].value_counts())

22
['DD.MM.YYYY' 'MM.YYYY' 'YYYY-MM-DD']
data_format
DD.MM.YYYY    20
MM.YYYY        1
YYYY-MM-DD     1
Name: count, dtype: int64


In [43]:
# create table for date rules
date_rules = datetypes[[
    "Form",
    "form_type",
    "record_designation",
    "dataelement_designation",
    "source_id",
    "data_format"
]].copy()

In [44]:
# add columns for rules (some boolean some string) with preset values

date_rules["notBeforeDOB"] = True
date_rules["notFuture"] = True
date_rules["notAfterDeath"] = True # optional if date of death exists (keep?)
date_rules["notAfterEpisode"] = date_rules["form_type"].str.strip().str.lower()=="longitudinal" # sets True for notAfterEpisode for all longidutinal variables
date_rules["notBeforeVariable"] = False
date_rules["notBeforeVariable_source_id"] = None
date_rules["notAfterVariable"] = False
date_rules["notAfterVariable_source_id"] = None


date_rules.to_csv("date_rules.csv", index=False, sep=";")

In [45]:
### RULE CHECKS

date_rule_checks=export_long.merge(
    date_rules, on="source_id", how="inner"
)



### Parsing values to datetime

In [46]:
# converting Dates to pandas datetime according to the formats given by data_format, in order to compare them

#without including "format" in to_datetime, Pandas will try to infer the format automatically which might lead to errors and is also slower.

# creates a mapping for parsing different formats
# considering dates without days to be beginning of month, adds 01 to the beginning to make dates compareable
def get_date_parse_config(data_format):
    configs = {
        "DD.MM.YYYY": {
            "parser_format": "%d.%m.%Y",
            "prefix": None,
        },
        "YYYY-MM-DD": {
            "parser_format": "%Y-%m-%d",
            "prefix": None,
        },
        "MM.YYYY": {
            "parser_format": "%d.%m.%Y",
            "prefix": "01.",
        },
        "MM-YYYY": {
            "parser_format": "%d-%m-%Y",
            "prefix": "01-",
        },
        "MM-DD-YYYY": {
            "parser_format": "%m-%d-%Y",
            "prefix": None,
        },
    }

    return configs.get(str(data_format).strip())

# parses dates and converts to pandas datetime format
def parse_dates(df):
    df = df.copy()
    df["source_value_parsed"] = pd.NaT
    df["data_format"] = df["data_format"].astype(str).str.strip()

    for data_format, group in df.groupby("data_format", dropna=False):
        config = get_date_parse_config(data_format)

        if config is None:
            continue

        values = group["source_value"].astype(str).str.strip()

        if config["prefix"] is not None:
            values = config["prefix"] + values

        parsed = pd.to_datetime(
            values,
            format=config["parser_format"],
            errors="coerce"
        )

        df.loc[group.index, "source_value_parsed"] = parsed

    return df


# parses values from Episode Date (format always DD/MM/YYYY)

def parse_episode_date(df):
    df = df.copy()

    df["episode_date_parsed"] = pd.to_datetime(
        df["Episode_Date"].astype(str).str.strip(),
        format="%d/%m/%Y",
        errors="coerce"
    )

    return df


date_rule_checks = parse_dates(date_rule_checks)
date_rule_checks = parse_episode_date(date_rule_checks)

# Rule Checks
### Not before episode / visit
### Not before DOB
### Not after Death

In [47]:
# Check if dates in longitudinal forms are after episode date

not_after_episode_issues = date_rule_checks[(date_rule_checks["notAfterEpisode"]==True) &
                                            date_rule_checks["source_value_parsed"].notna() &
                                            date_rule_checks["episode_date_parsed"].notna() &
                                            (date_rule_checks["source_value_parsed"] > date_rule_checks["episode_date_parsed"])
                                            ]

not_after_episode_issues

,Unnamed: 0,PID,Episode,Episode_Date,IX,source_id,source_value,Form,form_type,record_designation,...,notBeforeDOB,notFuture,notAfterDeath,notAfterEpisode,notBeforeVariable,notBeforeVariable_source_id,notAfterVariable,notAfterVariable_source_id,source_value_parsed,episode_date_parsed
1630,42848,12,2,31/03/2021,1,X_228_47_197,01.04.2021,Data at visit,Longitudinal,Visit Date,...,True,True,True,True,False,None,False,None,2021-04-01,2021-03-31
1640,42858,127,1,25/10/2021,1,X_228_47_197,26.10.2021,Data at visit,Longitudinal,Visit Date,...,True,True,True,True,False,None,False,None,2021-10-26,2021-10-25
1641,42859,128,1,26/10/2021,1,X_228_47_197,26.11.2021,Data at visit,Longitudinal,Visit Date,...,True,True,True,True,False,None,False,None,2021-11-26,2021-10-26
1928,43146,44,2,20/09/2021,1,X_228_47_197,22.09.2021,Data at visit,Longitudinal,Visit Date,...,True,True,True,True,False,None,False,None,2021-09-22,2021-09-20
1951,43169,57,2,09/08/2021,1,X_228_47_197,19.08.2021,Data at visit,Longitudinal,Visit Date,...,True,True,True,True,False,None,False,None,2021-08-19,2021-08-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14204,247200,245,1,15/11/2022,1,X_236_0_344,07.12.2022,Status,Longitudinal,NaN,...,True,True,True,True,False,None,False,None,2022-12-07,2022-11-15
14442,247438,31,6,04/10/2022,1,X_236_0_344,27.11.2022,Status,Longitudinal,NaN,...,True,True,True,True,False,None,False,None,2022-11-27,2022-10-04
14554,247550,56,1,21/07/2021,1,X_236_0_344,21.09.2021,Status,Longitudinal,NaN,...,True,True,True,True,False,None,False,None,2021-09-21,2021-07-21
14586,247582,63,2,28/09/2021,1,X_236_0_344,21.10.2021,Status,Longitudinal,NaN,...,True,True,True,True,False,None,False,None,2021-10-21,2021-09-28


In [48]:
# Check which variables raised issues

print(not_after_episode_issues["dataelement_designation"].unique())
not_after_episode_issues.to_csv("not_after_episode_issues.csv", sep=";", index=False)

['Visit Date' 'Date of HVPG' 'Physical examination date'
 'Date of Collection' 'Date of arterial blood sample  collection'
 'ICU admission date:' 'Echocardiography date' 'Inclusion Date'
 'Collection date' 'Date of Death' 'Date of liver transplantation']


### Not before Visit

In [49]:
# Re-checking with Date of Visit instead of Episode Date

DATE_OF_VISIT = "X_228_47_197"

visit_dates = date_rule_checks.loc[
    date_rule_checks["source_id"] == DATE_OF_VISIT, ["PID", "Episode", "source_value_parsed"]].rename(columns={"source_value_parsed":"visit_date_parsed"})

date_rule_checks = date_rule_checks.merge(visit_dates, on=["PID", "Episode"], how= "left",  validate="many_to_one"
)


not_after_visit_issues = date_rule_checks[(date_rule_checks["notAfterEpisode"]==True) &
                                            date_rule_checks["source_value_parsed"].notna() &
                                            date_rule_checks["visit_date_parsed"].notna() &
                                            (date_rule_checks["source_value_parsed"] > date_rule_checks["visit_date_parsed"])
                                            ]

print(not_after_visit_issues["dataelement_designation"].unique())
not_after_visit_issues.to_csv(f"{OUTPUT_DIR}/not_after_visit_issues.csv", sep=";", index=False)

['Date of HVPG' 'Physical examination date' 'Date of Collection'
 'Date of arterial blood sample  collection' 'ICU admission date:'
 'Echocardiography date' 'Inclusion Date' 'Collection date'
 'Date of Death' 'Date of liver transplantation']


In [50]:
# reorders columns
cols = [
    "PID",
    "Episode",
    "record_designation",
    "dataelement_designation",
    "episode_date_parsed",
    "visit_date_parsed",
    "source_value_parsed",
    "source_id",
    "Episode_Date",
    "source_value",
    "data_format"
]

not_after_visit_issues_clean = not_after_visit_issues[cols]
not_after_visit_issues_clean.to_csv(f"{OUTPUT_DIR}/not_after_visit_issues_clean.csv", sep=";", index=False)
not_after_visit_issues_clean

,PID,Episode,record_designation,dataelement_designation,episode_date_parsed,visit_date_parsed,source_value_parsed,source_id,Episode_Date,source_value,data_format
2112,124,2,4. HVPG: hepatic-venous-pressure-gradient,Date of HVPG,2021-12-04,2021-12-04,2021-12-20,X_229_51_212,04/12/2021,20.12.2021,DD.MM.YYYY
2129,128,5,4. HVPG: hepatic-venous-pressure-gradient,Date of HVPG,2023-05-12,2023-05-12,2023-05-16,X_229_51_212,12/05/2023,16.05.2023,DD.MM.YYYY
2552,23,1,4. HVPG: hepatic-venous-pressure-gradient,Date of HVPG,2021-03-09,2021-03-09,2021-06-23,X_229_51_212,09/03/2021,23.06.2021,DD.MM.YYYY
3122,106,2,NaN,Physical examination date,2022-01-14,2022-01-14,2022-01-15,X_230_0_232,14/01/2022,15.01.2022,DD.MM.YYYY
3142,12,1,NaN,Physical examination date,2021-01-19,2021-01-19,2021-02-02,X_230_0_232,19/01/2021,02.02.2021,DD.MM.YYYY
...,...,...,...,...,...,...,...,...,...,...,...
14204,245,1,NaN,Date of liver transplantation,2022-11-15,2022-11-15,2022-12-07,X_236_0_344,15/11/2022,07.12.2022,DD.MM.YYYY
14442,31,6,NaN,Date of liver transplantation,2022-10-04,2022-10-04,2022-11-27,X_236_0_344,04/10/2022,27.11.2022,DD.MM.YYYY
14554,56,1,NaN,Date of liver transplantation,2021-07-21,2021-07-21,2021-09-21,X_236_0_344,21/07/2021,21.09.2021,DD.MM.YYYY
14586,63,2,NaN,Date of liver transplantation,2021-09-28,2021-09-28,2021-10-21,X_236_0_344,28/09/2021,21.10.2021,DD.MM.YYYY


### Not before DOB

In [51]:

DATE_OF_BIRTH = "X_225_0_55"

dates_of_birth = date_rule_checks.loc[
    date_rule_checks["source_id"] == DATE_OF_BIRTH, ["PID", "source_value_parsed"]
        ].dropna(subset=["source_value_parsed"]).drop_duplicates(subset=["PID"]).rename(columns={"source_value_parsed":"dob_parsed"})

date_rule_checks = date_rule_checks.merge(dates_of_birth, on=["PID"], how= "left")


not_before_dob_issues = date_rule_checks[(date_rule_checks["notBeforeDOB"]==True) &
                                            date_rule_checks["source_value_parsed"].notna() &
                                            date_rule_checks["dob_parsed"].notna() &
                                            (date_rule_checks["source_value_parsed"] < date_rule_checks["dob_parsed"])
                                            ]


# reorders columns
cols = [
    "PID",
    "dob_parsed",
    "Episode",
    "record_designation",
    "dataelement_designation",
    "source_value_parsed",
    "source_id",
    "source_value",
    "data_format"
]
not_before_dob_issues = not_before_dob_issues[cols]
print(not_before_dob_issues["dataelement_designation"].unique())
not_before_dob_issues.to_csv(f"{OUTPUT_DIR}/not_before_dob_issues.csv", sep=";", index=False)

['Date of diagnosis of cirrhosis']


### Not after Death



In [52]:
DATE_OF_DEATH = "X_236_0_339"

dates_of_death = date_rule_checks.loc[
    date_rule_checks["source_id"] == DATE_OF_DEATH, ["PID", "source_value_parsed"]
        ].dropna(subset=["source_value_parsed"]).drop_duplicates(subset=["PID"]).rename(columns={"source_value_parsed":"dod_parsed"})

date_rule_checks = date_rule_checks.merge(dates_of_death, on=["PID"], how= "left")


not_after_death_issues = date_rule_checks[(date_rule_checks["notAfterDeath"]==True) &
                                            date_rule_checks["source_value_parsed"].notna() &
                                            date_rule_checks["dod_parsed"].notna() &
                                            (date_rule_checks["source_value_parsed"] > date_rule_checks["dod_parsed"])
                                            ]


# reorders columns
cols = [
    "PID",
    "dod_parsed",
    "Episode",
    "record_designation",
    "dataelement_designation",
    "source_value_parsed",
    "source_id",
    "source_value",
    "data_format"
]
not_after_death_issues = not_after_death_issues[cols]
print(not_after_death_issues["dataelement_designation"].unique())
not_after_death_issues.to_csv(f"{OUTPUT_DIR}/not_after_death_issues.csv", sep=";", index=False)

['Physical examination date' 'Date of Collection'
 'Date of arterial blood sample  collection']


### Not in Future

In [53]:
today = pd.Timestamp.today().normalize() # Todays date without time, current timezone



not_in_future_issues = date_rule_checks[(date_rule_checks["notFuture"]==True) &
                                            date_rule_checks["source_value_parsed"].notna() &
                                            (date_rule_checks["source_value_parsed"] > today)
                                            ]


# reorders columns
cols = [
    "PID",
    "Episode",
    "record_designation",
    "dataelement_designation",
    "source_value_parsed",
    "source_id",
    "source_value",
    "data_format"
]
not_in_future_issues = not_in_future_issues[cols]
print(not_in_future_issues["dataelement_designation"].unique())
not_in_future_issues.to_csv(f"{OUTPUT_DIR}/not_in_future_issues.csv", sep=";", index=False)

['If listed-listed liver transplantation date']
